# Wooplix Proposal Agent — Google Colab

**Wooplix Technologies Private Limited** — Together, We Achieve More

Give the agent a customer requirement (pasted **text** or uploaded **DOCX/PDF/TXT**). It learns from your **previous proposals in Zoho CRM**, drafts a new proposal, merges it into your **Wooplix Zoho Writer template**, and saves a review-ready **DOCX/PDF to Zoho WorkDrive**.

The notebook runs in three clean phases — **collect everything first, then generate**:

| Phase | What happens |
|---|---|
| **0 — Setup** | Install packages, load secrets, fetch Zoho token, test connections |
| **1 — Inputs** | Logo, customer requirement (text/file), previous proposals from CRM |
| **2 — Analyze** | Groq LLM drafts the structured proposal using CRM precedent |
| **3 — Generate** | Build DOCX, merge into Writer template, save DOCX/PDF to WorkDrive |

**Rules:** the LLM never marks a proposal APPROVED; effort/pricing are proposed only when CRM precedent supports them and are always labeled with the source deal; **all credentials live in Colab Secrets**.

### 🔐 Security note (read first)

An earlier version of this notebook had a Zoho refresh token and template URL pasted into code. Treat those as exposed: **revoke/rotate the refresh token in Zoho**, then store the new value only in Colab Secrets.

The refresh token needs these scopes: `ZohoCRM.modules.READ`, `ZohoWriter.documentEditor.ALL`, `ZohoWriter.merge.ALL`, `WorkDrive.files.ALL`.

## Phase 0 — Setup (run once per session)

Collects every credential and verifies all connections before any work begins.

In [ ]:
# 0. Install packages FIRST.
# This must be the first code cell because imports in a fresh Colab runtime can otherwise fail.
!pip -q install -U groq python-docx pypdf requests

import os
import json
import re
import time
import textwrap
from pathlib import Path

from google.colab import files, userdata
from IPython.display import display, Image
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from pypdf import PdfReader
from groq import Groq

print("✅ Packages installed and imported successfully.")


### 0.1 Configure secrets

In Colab, click the **🔑 Secrets** panel and add (exact names):

| Secret | Required | Purpose |
|---|---|---|
| `GROQ_API_KEY` | yes | Groq LLM |
| `GROQ_MODEL` | no | defaults to `openai/gpt-oss-120b` |
| `ZOHO_CLIENT_ID` | yes | Zoho OAuth |
| `ZOHO_CLIENT_SECRET` | yes | Zoho OAuth |
| `ZOHO_REFRESH_TOKEN` | yes* | *fetched automatically by 0.2 if absent |
| `ZOHO_WRITER_TEMPLATE_ID` | recommended | proposal template (or paste when prompted) |
| `ZOHO_CRM_MODULE` | no | module holding past proposals (default `Deals`) |
| `ZOHO_CRM_VERSION` | no | CRM API version (default `v2`) |
| `ZOHO_WORKDRIVE_FOLDER_ID` | no | WorkDrive folder for output (default `root`) |

In [ ]:
# 2. Company config + load ALL secrets (nothing hardcoded).
COMPANY_NAME = "Wooplix Technologies Private Limited"
COMPANY_TAGLINE = "Together, We Achieve More"
COMPANY_WEBSITE = "https://www.wooplix.com/"
COMPANY_EMAIL = "enquiry@wooplix.com"

# Change only if your Zoho org is in a different data center.
ZOHO_ACCOUNTS_URL = "https://accounts.zoho.in"

def get_secret(name, required=True, default=""):
    try:
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name, default)
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret '{name}'. Open the 🔑 Secrets panel, add it, then rerun this cell."
        )
    return value or default

# --- Groq ---
GROQ_API_KEY = get_secret("GROQ_API_KEY")
GROQ_MODEL = get_secret("GROQ_MODEL", required=False) or "openai/gpt-oss-120b"

# --- Zoho OAuth (all three required) ---
ZOHO_CLIENT_ID = get_secret("ZOHO_CLIENT_ID")
ZOHO_CLIENT_SECRET = get_secret("ZOHO_CLIENT_SECRET")
ZOHO_REFRESH_TOKEN = get_secret("ZOHO_REFRESH_TOKEN", required=False)  # filled by bootstrap cell 2B if absent

# --- Zoho Writer template: from Secrets, or paste a full template URL / id ---
ZOHO_WRITER_TEMPLATE_ID = get_secret("ZOHO_WRITER_TEMPLATE_ID", required=False)
_m = re.search(r"/template/([A-Za-z0-9]+)", ZOHO_WRITER_TEMPLATE_ID)
if _m:
    ZOHO_WRITER_TEMPLATE_ID = _m.group(1)

print("✅ Secrets loaded.")
print("  Groq model:", GROQ_MODEL)
print("  Zoho creds present:", bool(ZOHO_CLIENT_ID), bool(ZOHO_CLIENT_SECRET), bool(ZOHO_REFRESH_TOKEN))
if not ZOHO_REFRESH_TOKEN:
    print("  ⚠️ No refresh token yet — run the next cell (2B) once to fetch it.")
if ZOHO_WRITER_TEMPLATE_ID:
    print("  Writer template id:", ZOHO_WRITER_TEMPLATE_ID)
else:
    print("  ⚠️ No ZOHO_WRITER_TEMPLATE_ID — Phase 3.2 (Writer merge) will be skipped.")
    print("     You still get the branded DOCX/PDF + WorkDrive upload. Add the id later to enable merge.")
print("  OAuth data center:", ZOHO_ACCOUNTS_URL)

In [ ]:
# 2B. One-time Zoho refresh-token bootstrap.
# Zoho requires ONE human approval click; after that this cell fetches the token itself.
# If ZOHO_REFRESH_TOKEN is already in Secrets, this cell does nothing.
ZOHO_SCOPES = "ZohoCRM.modules.READ,ZohoWriter.documentEditor.ALL,ZohoWriter.merge.ALL,WorkDrive.files.ALL"

if ZOHO_REFRESH_TOKEN:
    print("✅ ZOHO_REFRESH_TOKEN already set from Secrets — skipping bootstrap.")
else:
    print("No refresh token yet. Two quick steps:\n")
    print("STEP 1 — Get a one-time grant code.")
    print("  Option A (Self Client, easiest): Zoho API Console -> your client -> 'Self Client' tab.")
    print("  Paste this scope string, pick a time duration, click Create, copy the Grant Code:\n")
    print("   ", ZOHO_SCOPES, "\n")
    print("  Option B (if you registered a Redirect URI): open this URL, approve, then copy the")
    print("  'code' parameter from the address bar of the redirect page:\n")
    auth_url = (
        ZOHO_ACCOUNTS_URL.rstrip("/") + "/oauth/v2/auth"
        + "?scope=" + ZOHO_SCOPES
        + "&client_id=" + ZOHO_CLIENT_ID
        + "&response_type=code"
        + "&access_type=offline"
        + "&prompt=consent"
    )
    print("   ", auth_url, "\n")

    grant_code = input("STEP 2 — Paste the grant code here: ").strip()
    if not grant_code:
        raise RuntimeError("A grant code is required to obtain the refresh token.")

    resp = requests.post(
        ZOHO_ACCOUNTS_URL.rstrip("/") + "/oauth/v2/token",
        params={
            "code": grant_code,
            "client_id": ZOHO_CLIENT_ID,
            "client_secret": ZOHO_CLIENT_SECRET,
            "grant_type": "authorization_code",
        },
        timeout=30,
    )
    if not resp.ok:
        raise RuntimeError(f"Grant-code exchange failed ({resp.status_code}): {resp.text[:1000]}")

    ZOHO_REFRESH_TOKEN = resp.json().get("refresh_token", "")
    if not ZOHO_REFRESH_TOKEN:
        raise RuntimeError("Exchange succeeded but returned no refresh_token: " + resp.text[:500])

    print("\n✅ Refresh token obtained for this session.")
    print("👉 ALSO save it in Colab Secrets as ZOHO_REFRESH_TOKEN so future runs skip this step:")
    print(ZOHO_REFRESH_TOKEN)

In [ ]:
# 3. Zoho access-token refresh (cached). No token is hardcoded — it comes from Secrets.
ZOHO_TOKEN_CACHE = {"token": None, "api_root": None, "expires_at": 0}

def get_zoho_token(force=False):
    if not ZOHO_REFRESH_TOKEN:
        raise RuntimeError("No Zoho refresh token. Run the bootstrap cell (2B) once, then rerun.")
    now = time.time()
    if (not force) and ZOHO_TOKEN_CACHE["token"] and now < ZOHO_TOKEN_CACHE["expires_at"] - 60:
        return ZOHO_TOKEN_CACHE["token"], ZOHO_TOKEN_CACHE["api_root"]

    client_id = ZOHO_CLIENT_ID
    if client_id and not client_id.startswith("1000."):
        client_id = "1000." + client_id

    token_url = ZOHO_ACCOUNTS_URL.rstrip("/") + "/oauth/v2/token"
    resp = requests.post(
        token_url,
        params={
            "refresh_token": ZOHO_REFRESH_TOKEN,
            "client_id": client_id,
            "client_secret": ZOHO_CLIENT_SECRET,
            "grant_type": "refresh_token",
        },
        timeout=30,
    )
    if not resp.ok:
        raise RuntimeError(f"Zoho token refresh failed ({resp.status_code}): {resp.text[:1000]}")

    payload = resp.json()
    token = payload.get("access_token")
    if not token:
        raise RuntimeError("Zoho response had no access_token: " + json.dumps(payload))

    api_root = (payload.get("api_domain") or "https://www.zohoapis.in").rstrip("/")
    ZOHO_TOKEN_CACHE.update({
        "token": token,
        "api_root": api_root,
        "expires_at": now + int(payload.get("expires_in", 3600)),
    })
    return token, api_root

# Names used by later cells:
ZOHO_ACCESS_TOKEN, ZOHO_RESOLVED_API_ROOT = get_zoho_token()
print("✅ Zoho access token refreshed.")
print("API domain:", ZOHO_RESOLVED_API_ROOT)

In [ ]:
# 4. Connection tests (Groq + Zoho Writer template). Does NOT send your customer document.
def test_groq_connection():
    client = Groq(api_key=GROQ_API_KEY)
    try:
        res = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": "Reply with exactly: WOOPLIX_TEST_OK"}],
            temperature=0.1,
            max_completion_tokens=32,
        )
        print("Groq response:", res.choices[0].message.content.strip())
    except Exception as e:
        raise RuntimeError("Groq connection failed. Check key/model.\n\n" + str(e))

def test_zoho_writer_template():
    token, api_root = get_zoho_token()
    url = f"{api_root}/writer/api/v1/templates/{ZOHO_WRITER_TEMPLATE_ID}"
    r = requests.get(url, headers={"Authorization": f"Zoho-oauthtoken {token}",
                                   "Accept": "application/json"}, timeout=30)
    if not r.ok:
        raise RuntimeError(
            f"Zoho Writer template check failed ({r.status_code}): {r.text[:1000]}\n"
            "Ensure the refresh token has ZohoWriter scopes and the template id is correct."
        )
    print("Writer template reachable:", r.json().get("name", ZOHO_WRITER_TEMPLATE_ID))

test_groq_connection()
if ZOHO_WRITER_TEMPLATE_ID:
    test_zoho_writer_template()
else:
    print("Writer template check skipped (no template id set).")
print("✅ Groq + Zoho Writer connections OK.")

## Phase 1 — Inputs (everything the agent needs)

Logo, the customer requirement, and your previous proposals as precedent.

### 1.1 Wooplix logo

The official registered logo is embedded below and written to `/content/wooplix_logo.png` — no upload needed.

In [ ]:
# 2A. Embed the official registered Wooplix logo inside the Colab runtime.
# The logo is embedded in this notebook so you do not need to upload it every time.
import base64
from IPython.display import display, Image

LOGO_PATH = "/content/wooplix_logo.png"
WOOPLIX_LOGO_B64 = "..."

with open(LOGO_PATH, "wb") as f:
    f.write(base64.b64decode(WOOPLIX_LOGO_B64))

display(Image(filename=LOGO_PATH, width=700))
print("✅ Embedded Wooplix logo is ready:", LOGO_PATH)


### 1.2 Customer requirement (text OR file)

Paste the requirement into `PASTED_REQUIREMENT`, or leave the placeholder and upload a DOCX/PDF/TXT.

In [ ]:
# 5. Requirement intake: pasted text OR uploaded file.
PASTED_REQUIREMENT = r"""
(Paste the customer's requirement text here, or leave this placeholder to upload a file instead.)
"""

def extract_docx_text(path):
    doc = Document(path)
    chunks = []
    for p in doc.paragraphs:
        if p.text.strip():
            chunks.append(p.text.strip())
    for table in doc.tables:
        for row in table.rows:
            row_cells = [c.text.strip() for c in row.cells]
            if any(row_cells):
                chunks.append(" | ".join(row_cells))
    return "\n".join(chunks)

def extract_pdf_text(path):
    reader = PdfReader(path)
    chunks = []
    for idx, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            chunks.append(f"[Page {idx}]\n{text}")
    return "\n\n".join(chunks)

def extract_requirement(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".pdf":
        return extract_pdf_text(path)
    if suffix == ".docx":
        return extract_docx_text(path)
    if suffix in (".txt", ".md"):
        return Path(path).read_text(errors="ignore")
    raise ValueError(f"Unsupported file type: {suffix}")

_placeholder = "(Paste the customer's requirement text here"
if PASTED_REQUIREMENT.strip() and _placeholder not in PASTED_REQUIREMENT:
    requirement_text = PASTED_REQUIREMENT.strip()
    print("✅ Using pasted requirement text.")
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No requirement provided (no text pasted, no file uploaded).")
    requirement_path = next(iter(uploaded))
    requirement_text = extract_requirement(requirement_path)
    print("✅ Requirement file:", requirement_path)

if not requirement_text.strip():
    raise RuntimeError("No text extracted. Scanned/image-only PDFs need OCR (not included).")

print(requirement_text[:20000])
print(f"\n✅ Extracted {len(requirement_text):,} characters.")

### 1.3 Previous proposals from Zoho CRM (precedent)

Pulls your recent CRM records so the new proposal reflects how you scoped and priced similar work before.

- Set `ZOHO_CRM_MODULE` to the module holding proposals (default `Deals`). Not sure? Run `list_crm_modules()`.
- If CRM is unreachable or missing scope, the run **continues without precedent** instead of failing.

In [ ]:
# 6. Pull previous proposals from Zoho CRM (precedent). Requires ZohoCRM.modules.READ scope.
CRM_API_VERSION = (get_secret("ZOHO_CRM_VERSION", required=False) or "v2").strip()
CRM_MODULE = (get_secret("ZOHO_CRM_MODULE", required=False) or "Deals").strip()
CRM_MAX_RECORDS = 25

def _crm_get(path, params=None):
    token, api_root = get_zoho_token()
    r = requests.get(
        f"{api_root}{path}",
        params=params or {},
        headers={"Authorization": f"Zoho-oauthtoken {token}", "Accept": "application/json"},
        timeout=60,
    )
    if r.status_code in (401, 403):
        raise RuntimeError(
            "Zoho CRM rejected the token (scope). Re-generate the refresh token with a CRM scope "
            "such as ZohoCRM.modules.READ (plus Writer + WorkDrive).\n" + r.text[:800]
        )
    return r

def list_crm_modules():
    r = _crm_get(f"/crm/{CRM_API_VERSION}/settings/modules", {"module_type": "all"})
    if not r.ok:
        raise RuntimeError(f"Could not list CRM modules ({r.status_code}): {r.text[:800]}")
    names = [m.get("api_name") for m in r.json().get("modules", []) if m.get("api_name")]
    print("Available CRM modules (set ZOHO_CRM_MODULE to one of these):")
    for n in names:
        print("  -", n)
    return names

def fetch_past_proposals(module=None, limit=CRM_MAX_RECORDS):
    module = module or CRM_MODULE
    if CRM_API_VERSION == "v2":
        path, params = f"/crm/v2/{module}", {"per_page": limit, "page": 1}
    else:
        path, params = f"/crm/{CRM_API_VERSION}/{module}/records", {"per_page": limit, "page": 1}
    r = _crm_get(path, params)
    if not r.ok:
        raise RuntimeError(f"CRM records fetch failed for module '{module}' ({r.status_code}): {r.text[:800]}")
    return r.json().get("data", []) or []

def _record_to_text(rec):
    lines = []
    for key, val in rec.items():
        if key.startswith("$") or val in (None, "", [], {}):
            continue
        if isinstance(val, dict):
            val = val.get("name") or val.get("value") or ""
        if isinstance(val, list):
            val = ", ".join(str(v.get("name", v) if isinstance(v, dict) else v) for v in val)
        text = str(val).strip()
        if len(text) > 1500:
            text = text[:1500] + "…"
        if text:
            lines.append(f"{key.replace('_', ' ').strip()}: {text}")
    return "\n".join(lines)

def build_reference_text(records, max_chars=60000):
    blocks = []
    for i, rec in enumerate(records, 1):
        body = _record_to_text(rec)
        if body.strip():
            blocks.append(f"--- PREVIOUS DEAL {i} ---\n{body}")
    ref = "\n\n".join(blocks)
    if len(ref) > max_chars:
        ref = ref[:max_chars] + "\n[Earlier deals truncated]"
    return ref

try:
    past_records = fetch_past_proposals()
    reference_text = build_reference_text(past_records)
    print(f"✅ Loaded {len(past_records)} previous CRM record(s) from module '{CRM_MODULE}' as precedent.")
    print("Preview:\n" + (reference_text[:2000] or "(no usable text fields found)"))
except Exception as e:
    reference_text = ""
    print("⚠️ Could not load previous proposals from Zoho CRM — continuing WITHOUT precedent.")
    print("   Reason:", str(e)[:500])
    print("   Tip: run list_crm_modules() to find the right module, then set ZOHO_CRM_MODULE in Secrets.")

## Phase 2 — Analyze & draft

The Proposal Solution Architect turns the requirement + precedent into structured proposal JSON.

### 2.1 Prompt

Strict: separates customer facts from assumptions, uses CRM precedent responsibly (labeling the source deal), never invents approved pricing.

In [ ]:
SYSTEM_PROMPT = r"""
You are the Proposal Solution Architect for Wooplix Technologies Private Limited.

Turn a customer requirement document into a Wooplix proposal written in Wooplix's house style: a precise scope-of-work, not a marketing document.

You may also receive PREVIOUS DEALS from Wooplix's Zoho CRM. Use them only as precedent for how similar work was scoped and, only if they contain usable rates or effort, for commercials. Never invent rates or hours.

HOUSE STYLE — match exactly how Wooplix sends proposals
- Project Introduction: ONE short paragraph. Pattern: "Wooplix Technologies Private Limited (Wooplix) will implement and customize <products> for <purpose> in line with the client's organizational policies and processes." Add at most one more sentence for the key business outcome.
- Project Scope: grouped by product. Under each product list named configuration / implementation areas (for example "User, Role & Access Configuration", "Organization Setup", "Expense Policy Configuration", "Approval Matrix Configuration", "<Product> Integration", "Reports & Export Templates"). Under each area put terse imperative bullets ending with a full stop ("Add and invite all users.", "Configure role-based access rights for data visibility and actions.").
- Carry the client's concrete values through verbatim wherever the requirement gives them: amounts, per-day or per-km rates, limits, bank names, user counts, report names, SLA figures. Never generalise them away.
- Mark genuinely optional items by appending "(If Required)" to the area name or the bullet.
- Pre-requisites: what the CLIENT must provide (credentials, master data, policies, sample templates, approval matrix, sign-offs).
- Deliverables: concrete bullets (configured module, integrations, export templates, UAT, training, post-deployment support with its duration).
- Plain professional English. No marketing adjectives, no emoji, no exclamation marks.
- Only include the sections defined in the JSON structure below. Do not add others.
- Do not state effort hours or pricing unless a Wooplix pricing/effort master or genuine precedent supplies them; otherwise leave commercials null.

NEVER
- Invent client requirements, products, integrations, rates, hours, dates or commitments.
- Mark the proposal APPROVED.
- Use AI-tell words: delve, leverage, cutting-edge, seamless, robust, holistic, synergy, transformative, empower, unlock, "in today's ... landscape", "it is important to note", furthermore/moreover as sentence openers, "in conclusion".

OUTPUT RULES
- Return ONLY valid JSON using the exact structure below.
{
  "client": { "company_name": "", "project_name": "", "contact": "" },
  "project_introduction": "",
  "scope": [
    { "product": "", "areas": [ { "area": "", "tasks": [""] } ] }
  ],
  "prerequisites": [""],
  "deliverables": [""],
  "open_points": [""],
  "commercials": null,
  "timeline": null,
  "status": "DRAFT"
}
- commercials: non-null ONLY when a pricing master or precedent supplies figures, as {"items":[{"item":"","amount":"","basis":""}], "total":"","note":""}; otherwise null.
- timeline: non-null ONLY when the requirement states an expectation or precedent supports one, as {"phases":[{"phase":"","duration":""}], "overall":""}; otherwise null.
- open_points: items to be finalized during discovery; empty list if none.
"""

In [ ]:
def parse_json_safely(raw):
    text = (raw or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            candidate = re.sub(r",\s*([}\]])", r"\1", text[start:end + 1])
            return json.loads(candidate)
        raise

def call_llm(requirement_text, reference_text=""):
    client = Groq(api_key=GROQ_API_KEY)

    max_chars = 320000
    truncated = len(requirement_text) > max_chars
    input_text = requirement_text[:max_chars]
    if truncated:
        input_text += "\n\n[DOCUMENT TRUNCATED: additional source text was not sent to the model.]"

    user_prompt = "CUSTOMER REQUIREMENT DOCUMENT\nAnalyze the following and return the required JSON.\n\n" + input_text
    if reference_text and reference_text.strip():
        user_prompt = (
            "PREVIOUS DEALS (precedent from Wooplix CRM — use only per the PRECEDENT RULES)\n\n"
            + reference_text.strip()
            + "\n\n"
            + user_prompt
        )

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_completion_tokens=16000,
        response_format={"type": "json_object"},
    )

    data = parse_json_safely(response.choices[0].message.content)
    if truncated:
        data.setdefault("review", {}).setdefault("missing_information", []).append(
            "The requirement document exceeded the input limit and was truncated."
        )
    return data

print("✅ LLM function ready.")

### 2.2 Run the analysis

In [ ]:
# 7. Run the AI analysis (requirement + CRM precedent).
proposal_data = call_llm(requirement_text, reference_text)
print(json.dumps(proposal_data, indent=2, ensure_ascii=False))

## Phase 3 — Generate deliverables

Build the DOCX, merge into your Writer template, and save DOCX/PDF to WorkDrive for review and sending.

### 3.1 Build the local proposal DOCX

In [ ]:
# 3.1 Build the proposal DOCX in Wooplix house style (scope-of-work layout).
import os
from datetime import datetime
from docx.shared import RGBColor

_INK = RGBColor(0x11, 0x11, 0x11)

def _ordinal_day(d):
    day = d.day
    suf = "th" if 11 <= day <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(day % 10, "th")
    return f"{day}{suf} {d.strftime('%b, %Y')}"

def _h(doc, text, size=13, before=10, after=4):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.bold = True
    r.font.size = Pt(size)
    r.font.color.rgb = _INK
    p.paragraph_format.space_before = Pt(before)
    p.paragraph_format.space_after = Pt(after)
    return p

def _sub(doc, text):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.bold = True
    r.font.size = Pt(11)
    r.font.color.rgb = _INK
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(2)
    return p

def _bullet(doc, text):
    p = doc.add_paragraph(str(text), style="List Bullet")
    for r in p.runs:
        r.font.size = Pt(10.5)
    p.paragraph_format.space_after = Pt(1)
    return p

def _para(doc, text, size=10.5):
    p = doc.add_paragraph()
    r = p.add_run(str(text))
    r.font.size = Pt(size)
    p.paragraph_format.space_after = Pt(6)
    return p

def create_proposal_docx(data, output="/content/Wooplix_Proposal_Draft.docx"):
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(10.5)

    client = data.get("client", {}) or {}
    client_name = client.get("company_name") or "Client"

    # Logo
    if os.path.exists(LOGO_PATH):
        lp = doc.add_paragraph()
        lp.add_run().add_picture(LOGO_PATH, width=Inches(2.0))
        lp.paragraph_format.space_after = Pt(2)
    _para(doc, f"{COMPANY_NAME}  |  {COMPANY_TAGLINE}", size=9)

    # Cover block: PRESENTED BY / PRESENTED TO / DATE
    tbl = doc.add_table(rows=1, cols=3)
    tbl.style = "Table Grid"
    cover = [
        ("PRESENTED BY:", COMPANY_NAME),
        ("PRESENTED TO:", client_name),
        ("DATE:", _ordinal_day(datetime.now())),
    ]
    for idx, (label, value) in enumerate(cover):
        cell = tbl.cell(0, idx)
        cell.paragraphs[0].clear()
        lr = cell.paragraphs[0].add_run(label)
        lr.bold = True
        lr.font.size = Pt(9)
        vp = cell.add_paragraph()
        vr = vp.add_run(value)
        vr.font.size = Pt(10)

    if client.get("project_name"):
        _h(doc, client["project_name"], size=16, before=12)

    # Project Introduction
    _h(doc, "Project Introduction", size=14)
    _para(doc, data.get("project_introduction") or "")

    # Project Scope grouped by product -> area -> imperative tasks
    _h(doc, "Project Scope", size=14)
    for prod in data.get("scope", []) or []:
        if prod.get("product"):
            _sub(doc, prod["product"], ) if False else _h(doc, prod["product"], size=12, before=8, after=2)
        for area in prod.get("areas", []) or []:
            _sub(doc, area.get("area", ""))
            for task in area.get("tasks", []) or []:
                if str(task).strip():
                    _bullet(doc, task)

    # Pre-requisites
    _h(doc, "Pre-requisites", size=14)
    _para(doc, "To begin the implementation, we will require:")
    for item in data.get("prerequisites", []) or []:
        if str(item).strip():
            _bullet(doc, item)

    # Deliverables
    _h(doc, "Deliverables", size=14)
    for item in data.get("deliverables", []) or []:
        if str(item).strip():
            _bullet(doc, item)

    # Optional: discovery open points
    open_points = data.get("open_points", []) or []
    if open_points:
        _h(doc, "Points to be Finalized During Discovery", size=14)
        for item in open_points:
            if str(item).strip():
                _bullet(doc, item)

    # Optional: indicative timeline
    tl = data.get("timeline") or None
    if tl:
        _h(doc, "Indicative Timeline", size=14)
        for ph in tl.get("phases", []) or []:
            _bullet(doc, f"{ph.get('phase','')} – {ph.get('duration','')}")
        if tl.get("overall"):
            _para(doc, f"Overall: {tl['overall']}")

    # Optional: commercials (only when a master/precedent supplied figures)
    com = data.get("commercials") or None
    if com:
        _h(doc, "Commercials", size=14)
        ct = doc.add_table(rows=1, cols=3)
        ct.style = "Table Grid"
        hdr = ct.rows[0].cells
        for c, label in zip(hdr, ["Item", "Amount", "Basis"]):
            c.paragraphs[0].clear()
            r = c.paragraphs[0].add_run(label)
            r.bold = True
            r.font.size = Pt(9.5)
        for it in com.get("items", []) or []:
            row = ct.add_row().cells
            row[0].text = str(it.get("item", ""))
            row[1].text = str(it.get("amount", ""))
            row[2].text = str(it.get("basis", ""))
        if com.get("total"):
            _para(doc, f"Total: {com['total']}")
        if com.get("note"):
            _para(doc, com["note"], size=9.5)

    # Footer line
    doc.add_paragraph()
    _para(doc, f"{COMPANY_NAME}  |  {COMPANY_EMAIL}  |  {COMPANY_WEBSITE}", size=9)

    doc.save(output)
    return output

print("✅ House-style proposal DOCX generator ready.")

In [ ]:
# 6. Generate and download the proposal.
output_file = create_proposal_docx(proposal_data)
print("✅ Proposal created:", output_file)
files.download(output_file)


### 3.2 Merge into the Zoho Writer template

The merge creates a **new** document from your template; the master template is never overwritten.

In [ ]:
# 7A. Writer template merge-field contract (house-style fields).
WRITER_FIELD_NAMES = [
    "ClientName", "ProjectName", "Contact",
    "ProjectIntroduction", "ScopeText", "Prerequisites", "Deliverables",
    "OpenPoints", "TimelineText", "CommercialsText", "ProposalStatus",
]
print("Create these merge-field names in your Zoho Writer template:")
for name in WRITER_FIELD_NAMES:
    print(" -", name)

In [ ]:
# 7E. Convert house-style proposal_data into flat Writer merge values.
def _lines(items):
    out = []
    for it in items or []:
        t = str(it).strip()
        if t:
            out.append("• " + t)
    return "\n".join(out) if out else "Not specified in the customer requirement."

def _scope_text(scope):
    blocks = []
    for prod in scope or []:
        lines = [str(prod.get("product", "")).strip()]
        for area in prod.get("areas", []) or []:
            lines.append("  " + str(area.get("area", "")).strip())
            for task in area.get("tasks", []) or []:
                if str(task).strip():
                    lines.append("    • " + str(task).strip())
        blocks.append("\n".join(l for l in lines if l.strip()))
    return "\n\n".join(blocks) if blocks else "Not specified in the customer requirement."

def _timeline_text(tl):
    if not tl:
        return "To be confirmed during discovery."
    parts = [f"{p.get('phase','')} – {p.get('duration','')}" for p in tl.get("phases", []) or []]
    if tl.get("overall"):
        parts.append(f"Overall: {tl['overall']}")
    return "\n".join("• " + p for p in parts) if parts else "To be confirmed during discovery."

def _commercials_text(com):
    if not com:
        return "To be confirmed; requires Wooplix pricing approval."
    rows = [f"{it.get('item','')}: {it.get('amount','')} ({it.get('basis','')})" for it in com.get("items", []) or []]
    if com.get("total"):
        rows.append(f"Total: {com['total']}")
    return "\n".join("• " + r for r in rows) if rows else "To be confirmed; requires Wooplix pricing approval."

def proposal_to_writer_record(data):
    client = data.get("client", {}) or {}
    return {
        "ClientName": client.get("company_name", ""),
        "ProjectName": client.get("project_name", ""),
        "Contact": client.get("contact", ""),
        "ProjectIntroduction": data.get("project_introduction", ""),
        "ScopeText": _scope_text(data.get("scope")),
        "Prerequisites": _lines(data.get("prerequisites")),
        "Deliverables": _lines(data.get("deliverables")),
        "OpenPoints": _lines(data.get("open_points")),
        "TimelineText": _timeline_text(data.get("timeline")),
        "CommercialsText": _commercials_text(data.get("commercials")),
        "ProposalStatus": data.get("status", "DRAFT"),
    }

writer_record = proposal_to_writer_record(proposal_data)
print(json.dumps(writer_record, indent=2, ensure_ascii=False)[:20000])

In [ ]:
# 7F. Merge into the Zoho Writer template and download DOCX + PDF.
#
# The Writer API merge operation takes merge_data whose keys must match
# the merge fields configured in the Writer template.
def merge_into_zoho_writer(template_id, record, access_token, api_root, filename_base):
    url = f"{api_root}/writer/api/v1/documents/{template_id}/merge"
    merge_data = json.dumps({"data": [record]}, ensure_ascii=False)

    downloaded = []

    for output_format, extension in [("docx", ".docx"), ("pdf", ".pdf")]:
        output_settings = json.dumps({
            "format": output_format,
            "filename": filename_base,
        })

        response = requests.post(
            url,
            headers={"Authorization": f"Zoho-oauthtoken {access_token}"},
            files={
                "output_settings": (None, output_settings, "application/json"),
                "merge_data": (None, merge_data, "application/json"),
            },
            timeout=180,
        )

        if not response.ok:
            raise RuntimeError(
                f"Zoho Writer merge failed ({response.status_code}): {response.text[:5000]}"
            )

        content_type = response.headers.get("content-type", "")
        local_path = "/content/" + filename_base + extension

        if "application/json" in content_type:
            raise RuntimeError(
                "Zoho Writer returned JSON instead of the merged file. "
                "Response: " + response.text[:5000]
            )

        with open(local_path, "wb") as fh:
            fh.write(response.content)

        downloaded.append(local_path)
        print(f"✅ Created in Zoho Writer merge: {local_path}")
        files.download(local_path)

    return downloaded

client_name = writer_record.get("ClientName") or "Client"
safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", str(client_name)).strip("_") or "Client"
writer_filename = f"Wooplix_Proposal_{safe_name}"

if ZOHO_WRITER_TEMPLATE_ID:
    merge_into_zoho_writer(
        ZOHO_WRITER_TEMPLATE_ID,
        writer_record,
        ZOHO_ACCESS_TOKEN,
        ZOHO_RESOLVED_API_ROOT,
        writer_filename,
    )
else:
    print("⚠️ Skipped Zoho Writer merge (no template id). Local DOCX/PDF + WorkDrive still produced.")


### 3.3 Save to Zoho WorkDrive

Uploads the merged DOCX/PDF so you can review and share (requires `WorkDrive.files.ALL`).

In [ ]:
# 7G. Upload merged outputs to Zoho WorkDrive for review/sharing.
WORKDRIVE_FOLDER_ID = get_secret("ZOHO_WORKDRIVE_FOLDER_ID", required=False) or "root"

def upload_to_workdrive(local_path, folder_id=None):
    folder_id = folder_id or WORKDRIVE_FOLDER_ID
    token, api_root = get_zoho_token()
    url = f"{api_root}/workdrive/api/v1/files/upload"
    with open(local_path, "rb") as fh:
        r = requests.post(
            url,
            headers={"Authorization": f"Zoho-oauthtoken {token}"},
            data={"folder_id": folder_id},
            files={"file": (os.path.basename(local_path), fh)},
            timeout=180,
        )
    if not r.ok:
        raise RuntimeError(f"WorkDrive upload failed ({r.status_code}): {r.text[:1000]}")
    info = (r.json().get("data") or [{}])[0]
    name = (info.get("attributes") or {}).get("name", os.path.basename(local_path))
    print(f"✅ Uploaded to WorkDrive: {name} (id={info.get('id')})")
    return info

for _p in [writer_filename + ".docx", writer_filename + ".pdf"]:
    _full = "/content/" + _p
    if os.path.exists(_full):
        try:
            upload_to_workdrive(_full)
        except Exception as e:
            print(f"⚠️ WorkDrive upload skipped for {_p}: {str(e)[:300]}")

### 3.4 Review & download the final draft

Inspect the generated document, then download the final populated draft to review before sending.

In [ ]:
import docx

def inspect_docx(file_path):
    print(f"🔍 Inspecting document: {file_path}")
    try:
        doc = docx.Document(file_path)
    except Exception as e:
        print(f"❌ Failed to load document: {e}")
        return

    print(f"Total paragraphs: {len(doc.paragraphs)}")
    print(f"Total tables: {len(doc.tables)}")

    print("\n--- First 10 paragraphs with text ---")
    count = 0
    for i, p in enumerate(doc.paragraphs):
        text = p.text.strip()
        if text:
            print(f"[{i}] {text[:200]}")
            count += 1
            if count >= 10:
                break

    if count == 0:
        print("⚠️ No paragraph text found!")

# Let's inspect both the locally generated draft and the merged file from Zoho
inspect_docx("/content/Wooplix_Proposal_Draft.docx")
print("\n" + "="*40 + "\n")
import os
merged_files = [f for f in os.listdir('/content') if f.startswith('Wooplix_Proposal_') and f.endswith('.docx')]
if merged_files:
    inspect_docx(os.path.join('/content', merged_files[0]))
else:
    print("❌ No merged Zoho DOCX file found in /content.")

In [ ]:
import shutil
from google.colab import files

# Copy our fully populated local draft to a clean final filename
src_path = "/content/Wooplix_Proposal_Draft.docx"
dest_path = "/content/Wooplix_Proposal_Final_Draft.docx"

shutil.copy(src_path, dest_path)
print(f"✅ Saved fully populated draft as: {dest_path}")

# Trigger browser download
files.download(dest_path)

## 8. Production architecture

```text
Customer Requirement
        ↓
Google Cloud Run
        ↓
Document Extraction
        ↓
LLM / Proposal Solution Architect
        ↓
Structured Proposal JSON
        ↓
Wooplix Effort Master + Pricing Master
        ↓
Wooplix Proposal Template
        ↓
DOCX / PDF
        ↓
Zoho WorkDrive / Zoho Writer
        ↓
Human Review / Approval
        ↓
Zoho CRM / Client Delivery
```

The LLM handles reasoning and drafting; deterministic application logic should control approved effort, pricing, approvals and final document rules.
